# Spoken dialogue: listen, think, speak

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/sds_demo.ipynb) [![Checked weekly](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/run_notebooks.yml)

Speech in, speech out, with a language model in between. ESPnet does the
listening and the speaking; the thinking is a small instruct model, because
that part is not ESPnet's job.

CPU throughout. A couple of minutes, most of it the first download.


## Install


In [ ]:
%pip install -q "espnet[tts]==202610.post1" espnet_model_zoo librosa transformers


## Somebody says something

A real recording rather than a synthesised one: a dialogue system meets
real voices, and reading back its own speech would flatter it.


In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O turn.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("turn.wav", sr=16000)
display(Audio(speech, rate=rate))


## Listen

OWSM-CTC, the same model as [`asr_demo.ipynb`](asr_demo.ipynb).


In [ ]:
from espnet2.bin.s2t_inference import Speech2Text

s2t = Speech2Text.from_pretrained("espnet/owsm_ctc_v4_1B", device="cpu")
heard = " ".join(
    text for _, _, text in s2t.decode_long(
        "turn.wav", lang_sym="<eng>", task_sym="<asr>"
    )
)
print("heard:", heard)


## Think

A 360M instruct model, small enough to answer on a CPU. This is the one
piece that is not ESPnet — `espnet2.sds.llm` wraps models like this so the
full system can switch between them, and the wrapper wants a Hugging Face
token, so the plain `transformers` pipeline is shorter here.


In [ ]:
from transformers import pipeline

chat = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct",
                device="cpu")

conversation = [
    {"role": "system", "content": "You are a helpful assistant in a spoken conversation. Reply to what the user just said in one short, natural sentence."},
    {"role": "user", "content": heard},
]
reply = chat(conversation, max_new_tokens=40)[0]["generated_text"][-1]["content"].strip()
print("reply:", reply)


## Speak

The same VITS as [`tts_demo.ipynb`](tts_demo.ipynb), reading the answer back.


In [ ]:
from espnet2.bin.tts_inference import Text2Speech

tts = Text2Speech.from_pretrained("espnet/kan-bayashi_ljspeech_vits")
wav = tts(reply)["wav"]
display(Audio(wav.view(-1).cpu().numpy(), rate=tts.fs))


## Where next

This is the cascade with the parts visible. The real thing is
[`egs2/TEMPLATE/sds1`](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/sds1):
a Gradio app you run locally, with a microphone, voice activity detection,
an end-to-end option beside the cascade, and latency and quality measured
while you talk.

- **The pieces**: `espnet2.sds.asr`, `espnet2.sds.llm`, `espnet2.sds.tts`
  wrap interchangeable models behind one interface
- **End-to-end**, no cascade: `espnet2.sds.end_to_end` runs Mini-Omni
- **A bigger brain**: any instruct model on the Hub goes in the `pipeline`
  above; 360M parameters is what a free CPU can answer with
